# How much of the prefix signal is run structure?

Registered test (H1-H4 committed before the run; see README). Train: 2185 stationary census rules; test: 162 fresh stationary rules, disjoint. Keys are the lengths of the first k runs of equal letters (R1 = lead alone), group-mean lookup with fallback to fewer runs (>= 3 training rules); reference is the 8-letter prefix lookup P8. Score: strict held-out R^2 against the training mean.

**Assumption.** Same stationarity filter and fresh-rule target as the W3c audit.

In [1]:
import json
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").exists())
d = json.loads((root / "experiments/archive/2026-10-kappa-prefix-runs/results/runs.json").read_text())
print("inputs:", {k: v[:12] for k, v in d["input_sha256"].items()}, "train", d["n_train"], "test", d["n_test"], "overlap", d["overlap"])
for k, v in d["r2"].items():
    lo, hi = d["ci"][k]
    print(f"{k:3s} R^2 {v:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]")
print(d["verdicts"])
assert d["overlap"] == 0
v = d["verdicts"]
assert [v[k] for k in v] == [False, False, False, True]  # H1-H3 failed, H4 held, as in the README
assert abs(d["r2"]["P8"] - 0.624) < 2e-3 and d["r2"]["R1"] < 0.05

inputs: {'probe_summary.csv': 'c76c7927185b', 'prefix_audit_predictions.csv': '742f0fe1d776'} train 2185 test 162 overlap 0
P8  R^2 +0.624  95% CI [+0.472, +0.732]
R1  R^2 -0.004  95% CI [-0.034, +0.013]
R2  R^2 +0.246  95% CI [+0.120, +0.369]
R3  R^2 +0.337  95% CI [+0.202, +0.482]
R4  R^2 +0.475  95% CI [+0.286, +0.627]
{'H1 R1 >= 0.15': False, 'H2 R2 >= 0.40': False, 'H3 R3 within 0.10 of P8': False, 'H4 no Rk exceeds P8 by > 0.05': True}


## What this shows and does not show
- **H1-H3 failed.** The lead alone has R^2 about 0; the first two runs give 0.25 and the first three 0.34, against 0.62 for the 8-letter prefix. H4 held: no run key beats P8.
- The unconditional lead is empty on these rules, consistent with the earlier finding that the lead-kappa association is at fixed area and is reproduced by conditioning on the area collider.
- The prefix signal is distributed over the opening (0.25, 0.34, 0.48 for two to four runs).
- **Not shown:** which letters at positions 5-8 matter; run keys and letter-prefix keys differ in group count and granularity and no matched control was run.

## Next
A registered per-position ablation of the 8-letter lookup with a group-count-matched control.